In [1]:
!pip install scrapling patchright msgspec nest-asyncio pandas openpyxl pydantic
!patchright install chromium
!patchright install-deps
!pip install playwright
!pip install browserforge

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.6/173.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 10.0 MB/s eta 0:00:00
  Attempting uninstall: lxml
    Found existing installation: lxml 6.1.0
    Uninstalling lxml-6.1.0:
      Successfully uninstalled lxml-6.1.0
184.3 MiB [                    ] 0% 0.0s184.3 MiB [                    ] 0% 124.1s184.3 MiB [                    ] 0% 381.8s184.3 MiB [                    ] 0% 436.7s184.3 MiB [                    ] 0% 352.2s184.3 MiB [                    ] 0% 304.0s184.3 MiB [                    ] 0% 274.4s184.3 MiB [                    ] 0% 253.9s184.3 MiB [                    ] 0% 238.1s184.3 MiB [                    ] 0% 223.2s184.3 MiB [                    ] 0% 207.2

In [2]:
from scrapling import StealthyFetcher
import time
import concurrent.futures
import re

def unified_job_scraper(keyword="AI Engineer", location="Egypt"):
    print(f"🔍 بدء عملية البحث عن: {keyword}\n" + "="*50)
    all_jobs = []

    # ---------------------------------------------------------
    # 1. Wuzzuf
    # ---------------------------------------------------------
    print("⏳ جاري سحب البيانات من Wuzzuf...")
    wuzzuf_url = f"https://wuzzuf.net/search/jobs/?q={keyword}&filters[post_date][0]=within_24_hours"
    try:
        response_wuzzuf = StealthyFetcher.fetch(wuzzuf_url, headless=True, network_idle=True)
        w_jobs = response_wuzzuf.css('div.css-pkv5jc, div.css-1gatmva')
        
        for job in w_jobs:
            # Title
            title_nodes = job.css('h2 a') or job.css('h2')
            title = title_nodes[0].text.strip() if title_nodes else "N/A"
            
            # Link - الحل الآمن لاستخراج الرابط
            link = "N/A"
            if title_nodes:
                raw_link = "N/A"
                if hasattr(title_nodes[0], 'attrib'):
                    raw_link = title_nodes[0].attrib.get('href', 'N/A')
                elif hasattr(title_nodes[0], 'attrs'):
                    raw_link = title_nodes[0].attrs.get('href', 'N/A')
                
                link = f"https://wuzzuf.net{raw_link}" if raw_link.startswith('/') else raw_link

            # Company (الآلية الذكية)
            company = "N/A"
            company_nodes = job.css('a.css-17s97q8, span.css-17s97q8, div.css-d7j1kk a, a.css-o171kl')
            if company_nodes:
                company = company_nodes[0].text.replace('-', '').strip()
                
            if company == "N/A" or company.lower() == title.lower():
                all_candidates = job.css('a, span')
                for node in all_candidates:
                    node_text = node.text.replace('-', '').strip()
                    if (node_text and node_text.lower() != title.lower() 
                        and node_text.lower() not in ['save', 'apply', 'view details', 'explore']):
                        company = node_text
                        break

            if title and title != "N/A":
                all_jobs.append({"Platform": "Wuzzuf", "Title": title, "Company": company, "Link": link})
    except Exception as e:
        print(f"❌ خطأ في Wuzzuf: {e}")

    # ---------------------------------------------------------
    # 2. LinkedIn
    # ---------------------------------------------------------
    print("⏳ جاري سحب البيانات من LinkedIn...")
    linkedin_url = f"https://www.linkedin.com/jobs/search?keywords={keyword}&location={location}&f_TPR=r86400"
    try:
        response_linkedin = StealthyFetcher.fetch(linkedin_url, headless=True, network_idle=True)
        time.sleep(2)
        l_jobs = response_linkedin.css('ul.jobs-search__results-list > li')
        
        for job in l_jobs:
            title_nodes = job.css('.base-search-card__title')
            title = title_nodes[0].text.strip() if title_nodes else "N/A"
            
            # Link - الحل الآمن
            link_nodes = job.css('a.base-card__full-link') or job.css('.base-search-card__title a') or job.css('a')
            link = "N/A"
            if link_nodes:
                raw_link = "N/A"
                if hasattr(link_nodes[0], 'attrib'):
                    raw_link = link_nodes[0].attrib.get('href', 'N/A')
                elif hasattr(link_nodes[0], 'attrs'):
                    raw_link = link_nodes[0].attrs.get('href', 'N/A')
                
                link = raw_link.split('?')[0] if raw_link != "N/A" else "N/A"
            
            # Company
            company_nodes = job.css('.base-search-card__subtitle a') or job.css('.base-search-card__subtitle')
            company = company_nodes[0].text.strip() if company_nodes else "N/A"
            
            title = re.sub(r'\s+', ' ', title)
            company = re.sub(r'\s+', ' ', company)
            
            if title and title != "N/A" and title != "":
                all_jobs.append({"Platform": "LinkedIn", "Title": title, "Company": company, "Link": link})
    except Exception as e:
        print(f"❌ خطأ في LinkedIn: {e}")

    

    # ---------------------------------------------------------
    # طباعة النتائج (Print Results)
    # ---------------------------------------------------------
    print("\n" + "="*70)
    print(f"✅ تم الانتهاء بنجاح! إجمالي الوظائف المكتشفة: {len(all_jobs)}")
    print("="*70)
    
    for idx, job in enumerate(all_jobs, 1):
        print(f"{idx}. [{job['Platform']}] 💼 {job['Title']} | 🏢 {job['Company']}")
        print(f"   🔗 Link: {job['Link']}\n")

    return all_jobs

def run_in_thread(keyword="AI Engineer", location="Egypt"):
    with concurrent.futures.ThreadPoolExecutor() as executor:
        future = executor.submit(unified_job_scraper, keyword, location)
        return future.result()
# اكتب التايتل و اللوكيشن هنا 
results = run_in_thread("AI Engineer", "Egypt")

🔍 بدء عملية البحث عن: AI Engineer
⏳ جاري سحب البيانات من Wuzzuf...


[2026-08-18 18:09:55] INFO: Fetched (200) <GET https://wuzzuf.net/search/jobs/?q=AI%20Engineer&filters[post_date][0]=within_24_hours> (referer: https://www.google.com/)


⏳ جاري سحب البيانات من LinkedIn...


[2026-08-18 18:09:59] INFO: Fetched (200) <GET https://www.linkedin.com/jobs/search?keywords=AI%20Engineer&location=Egypt&f_TPR=r86400&position=1&pageNum=0> (referer: https://www.google.com/)



✅ تم الانتهاء بنجاح! إجمالي الوظائف المكتشفة: 27
1. [LinkedIn] 💼 AI & Automation Engineer | 🏢 Ostoul Capital Group
   🔗 Link: https://eg.linkedin.com/jobs/view/ai-automation-engineer-at-ostoul-capital-group-4453084017

2. [LinkedIn] 💼 AI Automation Engineer | 🏢 Prestige marketing
   🔗 Link: https://eg.linkedin.com/jobs/view/ai-automation-engineer-at-prestige-marketing-4452810394

3. [LinkedIn] 💼 AI Engineer Team Leader | 🏢 ELSEWEDY ELECTRIC
   🔗 Link: https://eg.linkedin.com/jobs/view/ai-engineer-team-leader-at-elsewedy-electric-4455712902

4. [LinkedIn] 💼 AI/ML Manager | 🏢 onebank
   🔗 Link: https://eg.linkedin.com/jobs/view/ai-ml-manager-at-onebank-4455712448

5. [LinkedIn] 💼 Senior AI & Data Science Engineer | 🏢 CID Consulting
   🔗 Link: https://eg.linkedin.com/jobs/view/senior-ai-data-science-engineer-at-cid-consulting-4455536360

6. [LinkedIn] 💼 Specialist (Data Science) | 🏢 Raya Holding for Financial Investments
   🔗 Link: https://eg.linkedin.com/jobs/view/specialist-data-scienc